# Assignment 7: Clinical NLP with LLMs and Embeddings

Extract structured data from clinical notes using LLM prompt engineering, then build a semantic search system using sentence embeddings.

**Dataset:** 75 synthetic discharge summaries from [Asclepius-Synthetic-Clinical-Notes](https://huggingface.co/datasets/aisc-team-a1/Asclepius-Synthetic-Clinical-Notes) (Kweon et al., 2023) in `asclepius_notes.json`.

## Setup

In [ ]:
%pip install -q -r requirements.txt

%reset -f

In [ ]:
import os
import json
import random
import numpy as np
from dotenv import load_dotenv

os.makedirs("output", exist_ok=True)
load_dotenv()
print("Setup complete!")

### API Key

Part 1 requires an [OpenRouter](https://openrouter.ai) API key (OpenAI keys also work). Copy the example and fill in your key:

In [ ]:
%%bash
cp example.env .env
# Then edit .env with your actual key

Part 2 runs locally and does not need an API key.

### Helper Functions (do not modify)

In [ ]:
# --- LLM client setup (do not modify) ---

def get_client():
    """Initialize the LLM client based on available API keys."""
    from openai import OpenAI

    if os.environ.get("OPENROUTER_API_KEY"):
        client = OpenAI(
            api_key=os.environ["OPENROUTER_API_KEY"],
            base_url="https://openrouter.ai/api/v1",
        )
        return client, "openrouter"

    if os.environ.get("OPENAI_API_KEY"):
        return OpenAI(), "openai"

    raise ValueError(
        "No API key found. Set OPENROUTER_API_KEY or OPENAI_API_KEY in .env"
    )


def call_llm(prompt, provider, client):
    """Send a prompt to the LLM and return the response text."""
    model = "openai/gpt-4o-mini" if provider == "openrouter" else "gpt-4o-mini"
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": "You are a medical information extraction assistant."},
            {"role": "user", "content": prompt},
        ],
        temperature=0,
        max_tokens=500,
    )
    return response.choices[0].message.content


def get_device():
    """Detect the best available device for local model inference."""
    try:
        import torch
        if torch.cuda.is_available():
            return "cuda"
        if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
            return "mps"
    except ImportError:
        pass
    return "cpu"

### Load Data

In [ ]:
with open("asclepius_notes.json") as f:
    asclepius = json.load(f)

print(f"Loaded {len(asclepius)} synthetic clinical notes")
print(f"Keys: {list(asclepius[0].keys())}")

In [ ]:
print(asclepius[0]["note"][:500] + "...")

---

## Part 1: Clinical Entity Extraction

Use LLM prompt engineering to extract structured medical data from clinical notes.

In [ ]:
# Select 4 notes for extraction
random.seed(2026)
sample = random.sample(asclepius, 4)
notes_p1 = [s["note"] for s in sample]

print(f"Selected {len(notes_p1)} notes for extraction")
for i, n in enumerate(notes_p1, 1):
    print(f"\n--- Note {i} ({len(n)} chars) ---")
    print(n[:150] + "...")

### `build_prompt`

Build a prompt that instructs the LLM to extract structured data from a clinical note.

In [ ]:
# TODO: Implement build_prompt
# Requirements:
#   - Describe the extraction task clearly
#   - Specify the JSON output schema with these fields:
#     {"diagnosis": str, "medications": list, "lab_values": dict, "confidence": float}
#   - When few_shot=True, include 1-2 example input/output pairs
#   - Include the clinical note text
def build_prompt(note, few_shot=False):
    pass  # replace with your implementation

### `parse_json_response`

Extract a JSON object from LLM response text, which may contain markdown code fences or other wrapping.

In [ ]:
# TODO: Implement parse_json_response
# Requirements:
#   - Handle clean JSON strings (direct json.loads)
#   - Handle JSON wrapped in ```json ... ``` markdown blocks
#   - Find JSON within surrounding text (look for outermost { and })
#   - Return None if parsing fails
def parse_json_response(text):
    pass  # replace with your implementation

### `validate_response`

Check that a parsed response dict contains all required keys.

In [ ]:
# TODO: Implement validate_response
# Required fields: diagnosis, medications, lab_values, confidence
# Return True if all present, False otherwise
def validate_response(response):
    pass  # replace with your implementation

### `extract_entities`

Orchestrate the full extraction pipeline: get client, build prompt, call LLM, parse, validate, return.

In [ ]:
# TODO: Implement extract_entities
# Steps:
#   1. client, provider = get_client()
#   2. prompt = build_prompt(note, few_shot=few_shot)
#   3. raw = call_llm(prompt, provider=provider, client=client)
#   4. parsed = parse_json_response(raw)
#   5. Validate and return (return None if parsing or validation fails)
def extract_entities(note, few_shot=False):
    pass  # replace with your implementation

### Test extraction

In [ ]:
results_p1 = []
for i, note in enumerate(notes_p1, 1):
    result = extract_entities(note, few_shot=True)
    print(f"--- Note {i} ---")
    if result:
        print(json.dumps(result, indent=2))
        results_p1.append(result)
    else:
        print("Extraction failed")
    print()

### Save Part 1 results (do not modify)

In [ ]:
with open("output/extraction_results.json", "w") as f:
    json.dump(results_p1, f, indent=2)

print(f"Saved {len(results_p1)} extraction results to output/extraction_results.json")

---

## Part 2: Semantic Search

Build a semantic search system that finds clinical notes by meaning rather than keywords, using sentence embeddings and cosine similarity.

This part runs locally — no API key needed.

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer("all-MiniLM-L6-v2", device=get_device())
print(f"Model loaded on {get_device()}")

In [ ]:
# Use all 75 notes for the search corpus
notes_p2 = [n["note"] for n in asclepius]
print(f"{len(notes_p2)} notes in search corpus")

### `embed_notes`

Generate embeddings for a list of notes using the sentence transformer model.

In [ ]:
# TODO: Implement embed_notes
# Use model.encode(notes) — returns a numpy array of shape (n_notes, embedding_dim)
def embed_notes(notes):
    pass  # replace with your implementation

### `find_similar`

Search notes by meaning using cosine similarity.

In [ ]:
# TODO: Implement find_similar
# Steps:
#   1. Embed the query with model.encode([query])
#   2. Compute cosine_similarity(query_embedding, embeddings)
#   3. Sort by score descending
#   4. Return top_k results as [{"note": str, "score": float}, ...]
def find_similar(query, notes, embeddings, top_k=2):
    pass  # replace with your implementation

### Run the search pipeline

In [ ]:
embeddings = embed_notes(notes_p2)
print(f"Embeddings: {embeddings.shape}")

queries = [
    "heart attack symptoms",
    "infectious disease with fever",
    "respiratory illness",
]

for q in queries:
    print(f"\nQuery: '{q}'")
    results = find_similar(q, notes_p2, embeddings, top_k=2)
    for i, r in enumerate(results, 1):
        print(f"  {i}. (score: {r['score']:.3f}) {r['note'][:80]}...")

### Save Part 2 results (do not modify)

In [ ]:
search_results = find_similar("heart attack symptoms", notes_p2, embeddings, top_k=3)
with open("output/search_results.json", "w") as f:
    json.dump(search_results, f, indent=2)

print(f"Saved {len(search_results)} search results to output/search_results.json")

---

## Validation

In [ ]:
print("Run 'python -m pytest .github/tests/ -v' in your terminal to check your work.")